In [1]:
# ============================================================
# Notebook 01 : Bronze Hazard Layer
# India Flood Risk Platform
# ============================================================

# Standard Library
from pathlib import Path

# Data Handling
import pandas as pd
import numpy as np

# NetCDF Handling
import xarray as xr

# Display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

print("Libraries loaded successfully!")

Libraries loaded successfully!


                DOWNLOAD
                   │
        (manual for now)
                   │
                   ▼
         data/raw/era5/
                   │
         *.nc files (any year)
                   │
                   ▼
    Notebook 01 Bronze v2
                   │
     Automatically detects files
                   │
                   ▼
      Bronze Master Parquet
                   │
                   ▼
      Notebook 03 Silver Grid
                   │
                   ▼
      Notebook 04 Silver Events
                   │
                   ▼
      Notebook 05 Event Matching
                   │
                   ▼
        Future Gold Layer

In [2]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

BRONZE_DIR = DATA_DIR / "bronze"

ERA5_DIR = (
    RAW_DIR
    / "era5"
)

ERA5_FILES = sorted(
    ERA5_DIR.glob("*.nc")
)

print("Project Root :", PROJECT_ROOT)
print("ERA5 Folder :", ERA5_DIR)
print("NetCDF Files Found :", len(ERA5_FILES))

for file in ERA5_FILES:
    print(" •", file.name)

Project Root : d:\Work\Projects\India_Flood_Risk_Platform
ERA5 Folder : d:\Work\Projects\India_Flood_Risk_Platform\data\raw\era5
NetCDF Files Found : 16
 • ERA5_India_TotalPrecipitation_2000.nc
 • ERA5_India_TotalPrecipitation_2001.nc
 • ERA5_India_TotalPrecipitation_2002.nc
 • ERA5_India_TotalPrecipitation_2003.nc
 • ERA5_India_TotalPrecipitation_2004.nc
 • ERA5_India_TotalPrecipitation_2005.nc
 • ERA5_India_TotalPrecipitation_2006.nc
 • ERA5_India_TotalPrecipitation_2007.nc
 • ERA5_India_TotalPrecipitation_2008.nc
 • ERA5_India_TotalPrecipitation_2009.nc
 • ERA5_India_TotalPrecipitation_2010.nc
 • ERA5_India_TotalPrecipitation_2011.nc
 • ERA5_India_TotalPrecipitation_2012.nc
 • ERA5_India_TotalPrecipitation_2013.nc
 • ERA5_India_TotalPrecipitation_2014.nc
 • ERA5_India_TotalPrecipitation_2023.nc


In [3]:
# ============================================================
# Process One ERA5 Dataset
# ============================================================

def process_era5_dataset(ds):

    # Extract rainfall variable
    tp = ds["tp"]

    # Convert to DataFrame
    df = tp.to_dataframe().reset_index()

    # Rename columns
    df.rename(
        columns={
            "valid_time": "date",
            "tp": "rainfall_m"
        },
        inplace=True
    )

    # Convert metres to millimetres
    df["rainfall_mm"] = df["rainfall_m"] * 1000
    df.drop(columns="rainfall_m", inplace=True)

    return df

In [4]:
# ============================================================
# Process All ERA5 Files
# ============================================================

bronze_dfs = []

for file in ERA5_FILES:

    print(f"Processing {file.name}...")

    ds = xr.open_dataset(file)

    df = process_era5_dataset(ds)

    bronze_dfs.append(df)

    ds.close()

print(f"\nSuccessfully processed {len(bronze_dfs)} dataset(s).")

Processing ERA5_India_TotalPrecipitation_2000.nc...
Processing ERA5_India_TotalPrecipitation_2001.nc...
Processing ERA5_India_TotalPrecipitation_2002.nc...
Processing ERA5_India_TotalPrecipitation_2003.nc...
Processing ERA5_India_TotalPrecipitation_2004.nc...
Processing ERA5_India_TotalPrecipitation_2005.nc...
Processing ERA5_India_TotalPrecipitation_2006.nc...
Processing ERA5_India_TotalPrecipitation_2007.nc...
Processing ERA5_India_TotalPrecipitation_2008.nc...
Processing ERA5_India_TotalPrecipitation_2009.nc...
Processing ERA5_India_TotalPrecipitation_2010.nc...
Processing ERA5_India_TotalPrecipitation_2011.nc...
Processing ERA5_India_TotalPrecipitation_2012.nc...
Processing ERA5_India_TotalPrecipitation_2013.nc...
Processing ERA5_India_TotalPrecipitation_2014.nc...
Processing ERA5_India_TotalPrecipitation_2023.nc...

Successfully processed 16 dataset(s).


In [7]:
# ============================================================
# Combine All Years
# ============================================================

bronze = pd.concat(
    bronze_dfs,
    ignore_index=True
)

print("=" * 60)
print("BRONZE DATASET")
print("=" * 60)

print("Rows :", len(bronze))
print("Columns :", len(bronze.columns))

bronze.head()

BRONZE DATASET
Rows : 88567160
Columns : 5


,date,latitude,longitude,number,rainfall_mm
0,2000-01-01,37.0,68.00,0,0.0
1,2000-01-01,37.0,68.25,0,0.0
2,2000-01-01,37.0,68.50,0,0.0
3,2000-01-01,37.0,68.75,0,0.0
4,2000-01-01,37.0,69.00,0,0.0


In [10]:
# ============================================================
# Bronze Dataset Information
# ============================================================

print(bronze.info())

<class 'pandas.DataFrame'>
RangeIndex: 88567160 entries, 0 to 88567159
Data columns (total 5 columns):
 #   Column       Dtype         
---  ------       -----         
 0   date         datetime64[ns]
 1   latitude     float64       
 2   longitude    float64       
 3   number       int64         
 4   rainfall_mm  float32       
dtypes: datetime64[ns](1), float32(1), float64(2), int64(1)
memory usage: 3.0 GB
None


In [11]:
bronze.describe()

,date,latitude,longitude,number,rainfall_mm
count,88567160,8.856716e+07,8.856716e+07,88567160.0,8.856716e+07
mean,2008-07-11 22:17:15.991039232,2.153216e+01,8.300000e+01,0.0,3.800611e+00
min,2000-01-01 00:00:00,6.000000e+00,6.800000e+01,0.0,0.000000e+00
25%,2004-01-03 00:00:00,1.375000e+01,7.550000e+01,0.0,0.000000e+00
50%,2008-01-06 00:00:00,2.150000e+01,8.300000e+01,0.0,3.380775e-01
75%,2012-01-09 00:00:00,2.925000e+01,9.050000e+01,0.0,3.507137e+00
max,2023-12-31 00:00:00,3.800000e+01,9.800000e+01,0.0,6.811971e+02
std,NaN,9.040489e+00,8.732125e+00,0.0,9.087266e+00


In [12]:
# ============================================================
# Missing Value Analysis
# ============================================================

bronze.isna().sum()

date           0
latitude       0
longitude      0
number         0
rainfall_mm    0
dtype: int64

In [13]:
# ============================================================
# Duplicate Check
# ============================================================

print("Duplicate Rows :", bronze.duplicated().sum())

Duplicate Rows : 0


In [14]:
# ============================================================
# Data Types
# ============================================================

bronze.dtypes

date           datetime64[ns]
latitude              float64
longitude             float64
number                  int64
rainfall_mm           float32
dtype: object

In [15]:
# ============================================================
# Rainfall Statistics
# ============================================================

bronze["rainfall_mm"].describe()

count    8.856716e+07
mean     3.800611e+00
std      9.087266e+00
min      0.000000e+00
25%      0.000000e+00
50%      3.380775e-01
75%      3.507137e+00
max      6.811971e+02
Name: rainfall_mm, dtype: float64

In [16]:
# ============================================================
# Dataset Preview
# ============================================================

bronze.head()

,date,latitude,longitude,number,rainfall_mm
0,2000-01-01,37.0,68.00,0,0.0
1,2000-01-01,37.0,68.25,0,0.0
2,2000-01-01,37.0,68.50,0,0.0
3,2000-01-01,37.0,68.75,0,0.0
4,2000-01-01,37.0,69.00,0,0.0


In [17]:
# ============================================================
# Date Range
# ============================================================

print("Start Date :", bronze["date"].min())
print("End Date   :", bronze["date"].max())

Start Date : 2000-01-01 00:00:00
End Date   : 2023-12-31 00:00:00


In [18]:
# ============================================================
# Latitude Range
# ============================================================

print("Minimum Latitude :", bronze["latitude"].min())
print("Maximum Latitude :", bronze["latitude"].max())
# ============================================================
# Longitude Range
# ============================================================

print("Minimum Longitude :", bronze["longitude"].min())
print("Maximum Longitude :", bronze["longitude"].max())

Minimum Latitude : 6.0
Maximum Latitude : 38.0
Minimum Longitude : 68.0
Maximum Longitude : 98.0


In [19]:
# ============================================================
# Export Bronze Layer
# ============================================================

output_file = (
    BRONZE_DIR /
    "bronze_rainfall_master.parquet"
)

bronze.to_parquet(
    output_file,
    index=False
)

print("Bronze Layer exported successfully!")
print(output_file)

Bronze Layer exported successfully!
d:\Work\Projects\India_Flood_Risk_Platform\data\bronze\bronze_rainfall_master.parquet


In [20]:
# ============================================================
# Final Validation
# ============================================================

bronze_check = pd.read_parquet(output_file)

print("=" * 60)
print("BRONZE LAYER VALIDATION")
print("=" * 60)

print(f"Rows                : {len(bronze_check):,}")
print(f"Columns             : {bronze_check.shape[1]}")
print(f"Date Range          : {bronze_check['date'].min()} to {bronze_check['date'].max()}")
print(f"Missing Values      : {bronze_check.isna().sum().sum()}")
print(f"Duplicate Rows      : {bronze_check.duplicated().sum()}")

BRONZE LAYER VALIDATION
Rows                : 88,567,160
Columns             : 5
Date Range          : 2000-01-01 00:00:00 to 2023-12-31 00:00:00
Missing Values      : 0
Duplicate Rows      : 0


In [21]:
# ============================================================
# Engineering Summary
# ============================================================

print("=" * 60)
print("BRONZE HAZARD LAYER SUMMARY")
print("=" * 60)

print(f"NetCDF Files Processed : {len(ERA5_FILES)}")
print(f"Total Records          : {len(bronze):,}")
print(f"Variables              : {bronze.shape[1]}")
print(f"Start Date             : {bronze['date'].min()}")
print(f"End Date               : {bronze['date'].max()}")

print("\nOutput")
print("-----------------------------")
print(output_file)

BRONZE HAZARD LAYER SUMMARY
NetCDF Files Processed : 16
Total Records          : 88,567,160
Variables              : 5
Start Date             : 2000-01-01 00:00:00
End Date               : 2023-12-31 00:00:00

Output
-----------------------------
d:\Work\Projects\India_Flood_Risk_Platform\data\bronze\bronze_rainfall_master.parquet
